In [ ]:
from hashlib import sha256
import json
from pathlib import Path
import sys
from tempfile import TemporaryDirectory, mkdtemp

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    ROOT = ROOT / 'pyaxistreamer-master'
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    raise RuntimeError('pyaxistreamer-master またはその親を作業フォルダにしてください')
sys.path.insert(0, str(ROOT))
from streamer_drift_diffusion import SimulationConfig, run_simulation
from streamer_drift_diffusion.kinetics import DRY_AIR_MECHANISM_PATH
from streamer_drift_diffusion.plasma_chemistry import generate_reaction_table, load_reaction_table
import fipy
print('Python:', sys.executable)
print('FiPy:', fipy.__version__)

In [ ]:
# 使用するBOLSIG+形式の断面積ファイル。ここではPy3BOLOS付属のPhelpsデータを指定。
CROSS_SECTIONS = ROOT.parent / 'Py3bolos' / 'Phelps' / 'Phelps_N2-O2_20261026.txt'
TABLE_PATH = ROOT / 'outputdata_coefficients' / 'dry_air_v4_rates_10000td.json'
MECHANISM_PATH = DRY_AIR_MECHANISM_PATH
FIELD_TD = np.geomspace(0.01, 10000., 257)  # 0 Tdは生成時に追加される。
ENERGY_MAX_EV = 900.
REBUILD_TABLE = True  # 断面積・反応機構・電場範囲・温度・組成を変更したときはTrue。

# 係数表からBOLOSの移動度・拡散と乾燥空気の電子衝突反応係数を使う。
config = SimulationConfig(
    # Komuro et al. (2012), Fig. 3 and 5. Uniform 20 µm grid for an initial comparison.
    nr=400, nz=900, delta=20e-6, cathode_radius=40e-6, electrode_gap=13e-3,
    z0=12.9e-3,  # Keep the existing Gaussian seed 0.1 mm below the needle tip.
    chemistry_table=TABLE_PATH,
    voltage_mode='impulse', impulse_peak_voltage=24e3, impulse_time_scale=1.0,
    voltage_waveform_file=str(ROOT / 'streamer_drift_diffusion' / 'data' / 'komuro2012_fig5_24kv.txt'),
    adaptive_timestep=True, t_end=25e-9,
    dt_safety_factor=0.7,  # Komuro比較ケース。
    output_interval=5e-9, dt_max=1e-12,
    max_steps=100_000,
)
OUTPUT_BASE = ROOT / ('outputdata_photo' if config.adaptive_timestep else 'outputdata_staggered')
print(config)

In [ ]:
# 初回は係数表を生成し、次回は同じ入力で作成済みの表を再利用する。
if not CROSS_SECTIONS.is_file():
    raise FileNotFoundError(f'CROSS_SECTIONS に断面積ファイルを指定してください: {CROSS_SECTIONS}')
source_hash = sha256(CROSS_SECTIONS.read_bytes()).hexdigest()
if REBUILD_TABLE or not TABLE_PATH.is_file():
    print('Py3BOLOSで乾燥空気の係数表を生成しています:', CROSS_SECTIONS, flush=True)
    data = generate_reaction_table(
        CROSS_SECTIONS, FIELD_TD, temperature=config.temperature,
        energy_max=ENERGY_MAX_EV,
        mechanism_path=MECHANISM_PATH,
        nitrogen_fraction=config.nitrogen_fraction,
    )
    TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=TABLE_PATH.parent) as directory:
        pending = Path(directory) / TABLE_PATH.name
        pending.write_text(json.dumps(data, indent=2, allow_nan=False) + '\n', encoding='utf-8')
        pending.replace(TABLE_PATH)

table = load_reaction_table(TABLE_PATH, config)
if (table.mechanism is None or not table.mechanism.tabulated
        or table.metadata.get('source_sha256') != source_hash
        or table.metadata.get('mechanism_sha256') != sha256(MECHANISM_PATH.read_bytes()).hexdigest()
        or table.metadata.get('energy_max_ev') != ENERGY_MAX_EV
        or not np.array_equal(table.field_td[1:], FIELD_TD)):
    raise ValueError('係数表と入力条件が一致しません。REBUILD_TABLE=True にして設定セルから実行してください。')
print('係数表:', TABLE_PATH)
print(f'E/N: {table.field_td[0]:g} ～ {table.field_td[-1]:g} Td ({table.field_td.size}点)')

In [ ]:
OUTPUT_DIR = OUTPUT_BASE
if any(OUTPUT_DIR.glob('*.csv')):
    OUTPUT_DIR = Path(mkdtemp(prefix=f'{OUTPUT_BASE.name}_', dir=OUTPUT_BASE.parent))
print('CSV出力先:', OUTPUT_DIR, flush=True)
state = run_simulation(config, output_dir=OUTPUT_DIR)
print('maximum electron density [m^-3]:', state.ne.max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fields = [(state.ne, 'Electron density [m$^{-3}$]'),
          (state.phi, 'Potential [V]'), (state.abs_e, '|E|/N [Td]')]
for ax, (values, title) in zip(axes, fields):
    image = ax.pcolormesh(state.rh * 1e3, state.zh * 1e3, values.T, shading='nearest', cmap='jet')
    ax.set(xlabel='r [mm]', ylabel='z [mm]', title=title)
    fig.colorbar(image, ax=ax)
plt.show()
# 必要な場合: fig.savefig(ROOT / 'final_state.png', dpi=160)

描画は最終時刻の電子密度・電位・電場を使用します。動画を作る場合は、描画Notebookの入力先を実行セルに表示された `OUTPUT_DIR` に合わせてください。